# Análisis de journeys de Usuario
---

<div style="text-align: center">
    <img src="https://raw.githubusercontent.com/ljpiere/tpdata_python/main/images/w1s1_2.png" width="400">
</div>

## Agenda y Objetivos de Aprendizaje


1. Comprender los *user journey* 
2. Realizar un análisis de embudo usando SQL
3. Analizar retención con cohortes

## *User Journey*

---

Un *user journey* es la representación de un proceso multi-etapa visto desde la perspectiva del usuario (ej.: descubrimiento → evaluación → conversión → uso/retención).  
Sirve para conectar métricas de producto con impacto de negocio (tasa de conversión, *drop-off*, tiempo entre etapas, LTV).

**Ejercicio (discursivo):** Analiza 3 journeys y define etapas + eventos:
- **E-commerce:** visita → *view_item* → *add_to_cart* → *begin_checkout* → *purchase*  
- **Educación:** *landing* → registro → primera lección → completar módulo → certificación  
- **Servicios públicos:** ingresar a portal → solicitar turno → adjuntar documentos → pago → confirmación


<center>
  <img src="https://practicum-content.s3.us-west-1.amazonaws.com/new-markets/Data_refactored_V8/ES/DA_S4-Ch2-L1_infografia_img03.png" alt="User journey Funnel" style="max-width:100%; height:auto;" width="50%">
</center>

### Ejercicio práctico: Analisis de *user journey* en marketing (EDA)

*Sopnora* es una empresa de marketin que nota que está perdiendo clientes sin saber en qué etapa del proceso, así que decide analizar su base de datos de eventos para descubrirlo
Para ello, el equipo de datos extrae información del dataset **ecommerce_jan_2021**, que contiene los registros de comportamiento de los usuarios durante enero de 2021.Cada fila en esta tabla representa una acción del usuario dentro del sitio web, como ver un producto, agregarlo al carrito o completar una compra. En otras palabras, cada acción es un evento que forma parte del *journey del cliente*. 

#### 📘 Diccionario de datos: `ecommerce_jan_2021`

| Nombre de la columna | Descripción | Tipo de dato | Ejemplo | Notas / Uso en análisis |
|---------------------|-------------|--------------|---------|-------------------------|
| `event_date` | Fecha en la que ocurrió el evento (formato AAAA-MM-DD). | Fecha | 2021-01-01 | Útil para agrupar eventos por día o crear series de tiempo. |
| `event_timestamp` | Momento exacto del evento como timestamp UNIX en microsegundos. | Entero | 1609462211068644 | Se puede convertir a fecha y hora legible para ordenar eventos cronológicamente. |
| `event_name` | Tipo de evento o acción del usuario registrada. | Texto | session_start | Permite identificar acciones como compras, visitas o inicios de sesión. |
| `event_value_in_usd` | Valor monetario del evento en dólares (si aplica). | Numérico (decimal) | 12.99 | Usado en eventos de compra; puede estar vacío en eventos no transaccionales. |
| `user_id` | Identificador único del usuario (generalmente anonimizado). | Texto / Entero | 51300349748243300 | Sirve para rastrear comportamiento y frecuencia de sesiones. |
| `user_first_touch_timestamp` | Timestamp (microsegundos) del primer contacto del usuario con la app o sitio. | Entero | 1609462211068644 | Útil para análisis de cohortes y retención. |
| `revenue` | Ingreso total generado por el evento. | Numérico (decimal) | 0 | Puede complementar o duplicar `event_value_in_usd`. |
| `currency` | Código de moneda según estándar ISO 4217. | Texto | USD | Importante para análisis internacionales o conversiones de moneda. |
| `category` | Tipo de plataforma o categoría del producto. | Texto | mobile | Permite segmentar los datos (móvil vs web, por ejemplo). |
| `mobile_brand_name` | Marca del dispositivo móvil del usuario. | Texto | Samsung | Útil para análisis de rendimiento por marca. |
| `mobile_model_name` | Modelo específico del dispositivo del usuario. | Texto | <Other> | Segmentación por modelo; puede faltar en eventos web. |
| `operating_system` | Sistema operativo o entorno del usuario. | Texto | Web | Permite identificar diferencias de comportamiento por sistema o navegador. |
| `language` | Idioma configurado en el dispositivo o navegador. | Texto | en-gb | Indica preferencias de idioma o posibles datos faltantes. |
| `country` | País del usuario según IP o configuración. | Texto | Peru | Usado para análisis geográficos. |
| `city` | Ciudad o región del usuario. | Texto | Monterrey | Puede estar vacío o anonimizado por privacidad. |
| `stream_id` | ID interno para rastrear sesiones o flujos de eventos. | Entero | 2100450278 | Útil para combinar datos entre distintas tablas de eventos. |
| `platform` | Plataforma donde se originó el evento (web o app). | Texto | WEB | Permite comparar comportamiento entre plataformas. |



**Ejercicio:** 
- Revisa la tabla `ecommerce_jan_2021` y marca cuáles campos sirven para ordenar eventos, agrupar usuarios y calcular conversiones.
- Cuenta la cantidad de eventos y de usuarios únicos
- Muestra los eventos únicos del funnel y la cantidad de cada uno de ellos

**Query de referencia**

```sql
SELECT *
FROM ecommerce_jan_2021
LIMIT 10;
```

####  Validación de calidad de datos
Antes de construir consultas:
- **Duplicados:** ¿múltiples eventos idénticos?  
- **Faltantes:** ¿etapas ausentes (p. ej., carrito sin checkout)?  
- **Tiempos:** ¿timestamps fuera de orden o en el futuro?



```sql
-- Detectar duplicados de purchase
SELECT user_id, 
        event_timestamp, 
        count(*) as numero_registros
FROM ecommerce_jan_2021
WHERE event_name = 'purchase'
GROUP BY user_id, event_timestamp, event_name;
```

```sql 
-- 2)  analizar que cantidad de productos son agregados por cada usuario
SELECT 
user_id, 
count(*)
FROM ecommerce_jan_2021
WHERE event_name = 'add_to_cart'
GROUP BY user_id
ORDER BY count(*) DESC
```



```sql 
-- 3)  Eventos fuera de tiempo
SELECT *
FROM ecommerce_jan_2021
WHERE event_date > NOW();
```

```sql 
-- 4) Usuarios con carrito pero sin checkout/purchase
-- 4) Usuarios con carrito pero sin checkout/purchase
WITH cart_users AS (
  SELECT DISTINCT user_id FROM ecommerce_jan_2021 WHERE event_name='add_to_cart'
),
checkout_users AS (
  SELECT DISTINCT user_id FROM ecommerce_jan_2021 WHERE event_name='begin_checkout'
),
purchase_users AS (
  SELECT DISTINCT user_id FROM ecommerce_jan_2021 WHERE event_name='purchase'
),
data AS (
SELECT cu.user_id,
       (cu.user_id IN (SELECT * FROM checkout_users)) AS has_checkout,
       (cu.user_id IN (SELECT * FROM purchase_users)) AS has_purchase
FROM cart_users cu
)
SELECT * FROM data
WHERE
has_checkout = false
and has_purchase = true
ORDER BY user_id;
```

## Análisis de embudo de conversión


Para entender **cómo los usuarios avanzan (o abandonan)** a lo largo de su recorrido, es clave analizar el embudo de conversión.  
Este análisis nos permite observar, paso a paso, **qué proporción de usuarios logra avanzar entre eventos clave** y en qué etapas se concentran las mayores pérdidas. Más allá de contar usuarios, el embudo nos ayuda a **detectar fricciones, priorizar oportunidades de mejora y conectar el comportamiento del usuario con decisiones de negocio**.

A partir de este análisis surgen preguntas como:  
- ¿En qué etapa del journey se pierde la mayor cantidad de usuarios?  
- ¿El abandono es esperado o indica un posible problema de experiencia?  
- ¿Qué etapa tendría mayor impacto si lográramos mejorar su retención?  
- ¿Conviene optimizar todo el funnel o enfocarnos en un punto específico?


```sql
-- Filtrado por ventana temporal 
WITH base AS (
  SELECT *
  FROM ecommerce_jan_2021
  WHERE event_timestamp >= '2021-01-01'
    AND event_timestamp <  '2021-01-10'
), cte_session AS (
  SELECT DISTINCT user_id FROM base WHERE event_name = 'session_start'
), cte_view AS (
  SELECT DISTINCT user_id FROM base WHERE event_name = 'view_item'
), cte_cart AS (
  SELECT DISTINCT user_id FROM base WHERE event_name = 'add_to_cart'
), cte_checkout AS (
  SELECT DISTINCT user_id FROM base WHERE event_name = 'begin_checkout'
), cte_purchase AS (
  SELECT DISTINCT user_id FROM base WHERE event_name = 'purchase'
)
SELECT
  (SELECT COUNT(*) FROM cte_session)  AS session_users,
  (SELECT COUNT(*) FROM cte_view)     AS view_users,
  (SELECT COUNT(*) FROM cte_cart)     AS cart_users,
  (SELECT COUNT(*) FROM cte_checkout) AS checkout_users,
  (SELECT COUNT(*) FROM cte_purchase) AS purchase_users;
```

Como siguiente paso, el equipo de negocio nos solicita **cuantificar con mayor precisión dónde se están perdiendo los usuarios dentro del embudo**, específicamente entre las acciones de ver un producto, agregarlo al carrito e iniciar el checkout.  
El objetivo es conocer **qué porcentaje de usuarios abandona después de cada etapa**, para identificar si el mayor problema ocurre antes o después de agregar productos al carrito y así enfocar posibles mejoras en la experiencia del usuario.


```sql 
WITH cte_view AS (
  SELECT DISTINCT user_id
  FROM ecommerce_jan_2021
  WHERE event_name = 'view_item'
),
cte_cart AS (
  SELECT DISTINCT user_id
  FROM ecommerce_jan_2021
  WHERE event_name = 'add_to_cart'
),
cte_checkout AS (
  SELECT DISTINCT user_id
  FROM ecommerce_jan_2021
  WHERE event_name = 'begin_checkout'
)
SELECT
  (SELECT COUNT(*) FROM cte_view)     AS viewed_products,
  (SELECT COUNT(*) FROM cte_cart)     AS added_to_cart,
  (SELECT COUNT(*) FROM cte_checkout) AS started_checkout,
  ((SELECT COUNT(*) FROM cte_view) - (SELECT COUNT(*) FROM cte_cart)) * 100
      / NULLIF((SELECT COUNT(*) FROM cte_view), 0) AS dropoff_after_view_pct,
  ((SELECT COUNT(*) FROM cte_cart) - (SELECT COUNT(*) FROM cte_checkout)) * 100
      / NULLIF((SELECT COUNT(*) FROM cte_cart), 0) AS dropoff_after_cart_pct;
```

### Interpretación de resultados
- Identifica el **cuello de botella** (mayor caída).  
- Propón hipótesis (p. ej., fricción en checkout móvil, valor del carrito, medios de pago).  
- Plantea experimento A/B o UX research para esa etapa.


# Análisis de cohortes



Como parte del análisis, el equipo también nos solicita **ir más allá del comportamiento agregado** y entender cómo se comportan distintos grupos de usuarios a lo largo del tiempo.  
Para ello, realizaremos un **análisis de cohortes**, que nos permitirá comparar la retención de usuarios según el periodo en el que iniciaron su journey. De esta forma podremos responder si los usuarios más recientes se comportan mejor o peor que cohortes anteriores y detectar **cambios estructurales en la experiencia, el producto o el contexto** que no serían visibles al analizar solo métricas globales.


**Cohortes trimestrales**

```sql
/* PASO 1: Identificar la cohorte trimestral de cada cliente */
WITH cohortes AS (
  SELECT
    customer_id,
    DATE_TRUNC('quarter', MIN(signup_date)) AS cohorte_trimestre,
    tenure_months,
    exited
  FROM clientes_banco
  GROUP BY customer_id, tenure_months, exited
),
/* PASO 2: Calcular clientes retenidos por trimestre */
retencion AS (
  SELECT
    cohorte_trimestre,
    COUNT(*) AS clientes_iniciales,
    COUNT(CASE WHEN tenure_months >= 3 AND exited = FALSE THEN 1 END) AS retained_q1,
    COUNT(CASE WHEN tenure_months >= 6 AND exited = FALSE THEN 1 END) AS retained_q2,
    COUNT(CASE WHEN tenure_months >= 9 AND exited = FALSE THEN 1 END) AS retained_q3
  FROM cohortes
  GROUP BY cohorte_trimestre
)
/* PASO 3: Crear la tabla de retención con porcentajes */
SELECT
  TO_CHAR(cohorte_trimestre, 'YYYY-"Q"Q') AS cohorte,
  clientes_iniciales,
  ROUND(retained_q1::numeric / clientes_iniciales, 2) AS trimestre_1,
  ROUND(retained_q2::numeric / clientes_iniciales, 2) AS trimestre_2,
  ROUND(retained_q3::numeric / clientes_iniciales, 2) AS trimestre_3
FROM retencion
ORDER BY cohorte_trimestre;

```

**Cohortes mensuales**

```sql
/* PASO 1: Identificar la cohorte mensual de cada cliente */
WITH cohortes AS (
  SELECT
    customer_id,
    DATE_TRUNC('month', MIN(signup_date)) AS cohorte_mes,
    tenure_months,
    exited
  FROM clientes_banco
  GROUP BY customer_id, tenure_months, exited
),

/* PASO 2: Calcular clientes retenidos por periodo */
retencion AS (
  SELECT
    cohorte_mes,
    COUNT(*) AS clientes_iniciales,
    COUNT(CASE WHEN tenure_months >= 1 AND exited = FALSE THEN 1 END) AS mes_1,
    COUNT(CASE WHEN tenure_months >= 3 AND exited = FALSE THEN 1 END) AS mes_3,
    COUNT(CASE WHEN tenure_months >= 6 AND exited = FALSE THEN 1 END) AS mes_6
  FROM cohortes
  GROUP BY cohorte_mes
)

/* PASO 3: Crear la tabla de retención con porcentajes */
SELECT
  TO_CHAR(cohorte_mes, 'YYYY-MM') AS cohorte,
  clientes_iniciales,
  ROUND(mes_1::numeric / clientes_iniciales * 100, 2) AS retencion_mes_1,
  ROUND(mes_3::numeric / clientes_iniciales * 100, 2) AS retencion_mes_3,
  ROUND(mes_6::numeric / clientes_iniciales * 100, 2) AS retencion_mes_6
FROM retencion
ORDER BY cohorte_mes;

```

El análisis de cohortes nos permite observar patrones de comportamiento que no son evidentes en métricas agregadas, pero su verdadero valor está en **cómo interpretamos los resultados**. Al comparar cohortes a lo largo del tiempo, surgen preguntas clave para la toma de decisiones, especialmente cuando apoyamos el análisis con visualizaciones como un *heatmap*, que facilita identificar tendencias de retención y abandono.

Algunas preguntas de reflexión son:
- ¿Qué cohortes muestran una mejor retención a lo largo del tiempo y qué pudo haber influido en ese comportamiento?  
- ¿Se observa algún cambio claro en la retención a partir de una cohorte específica que coincida con un cambio en el producto o la estrategia?  
- Al visualizar el análisis mediante un **heatmap**, ¿qué patrones o anomalías se identifican rápidamente que no serían tan evidentes en una tabla numérica?


**Segmentación por localidad**

Finalmente, el equipo de negocio solicita **profundizar el análisis incorporando una segmentación por geografía**, con el objetivo de entender si el comportamiento de los usuarios varía según su ubicación.  
Al analizar el funnel y la retención por región o país, podremos identificar **diferencias estructurales en el journey**, detectar mercados con mayor potencial de mejora y evaluar si una misma estrategia debería aplicarse de forma homogénea o adaptarse por zona geográfica.


```sql
WITH cohortes AS (
	SELECT
		DATE_TRUNC('month', signup_date) AS cohorte_mes,
		geography,
		customer_id,
		tenure_months,
		exited
	FROM clientes_banco
),
retencion AS (
SELECT
	cohorte_mes,
	geography,
	COUNT(*) AS clientes_iniciales,
	COUNT(CASE WHEN tenure_months >= 1 AND exited = FALSE THEN 1 END) AS retained_m1,
	COUNT(CASE WHEN tenure_months >= 2 AND exited = FALSE THEN 1 END) AS retained_m2,
	COUNT(CASE WHEN tenure_months >= 3 AND exited = FALSE THEN 1 END) AS retained_m3
FROM cohortes
GROUP BY cohorte_mes, geography
)

SELECT
	TO_CHAR(cohorte_mes, 'Mon YYYY') AS cohorte,
	geography,
	ROUND(retained_m1::numeric / clientes_iniciales, 2) AS mes_1,
	ROUND(retained_m2::numeric / clientes_iniciales, 2) AS mes_2,
	ROUND(retained_m3::numeric / clientes_iniciales, 2) AS mes_3
FROM retencion
ORDER BY cohorte_mes, geography;

```

> 💡 Usamos `::numeric` para convertir el valor a tipo numérico y evitar divisiones enteras.  
> Esto asegura que el cálculo de la tasa de retención conserve los decimales antes de aplicar `ROUND()`, obteniendo proporciones correctas y comparables.


Ahora piensa en la siguientes preguntas:
- ¿Qué diferencias se observan en la retención temprana (mes 1 o 2) entre geografías y qué factores podrían explicarlas (cultura, mercado, producto, contexto)?  
- ¿Existen geografías donde la retención se mantenga alta en el tiempo y qué aprendizajes podrían extrapolarse a otras regiones?  
- Al visualizar las cohortes segmentadas por geography mediante un **heatmap**, ¿qué patrones regionales se identifican rápidamente y cómo podrían influir en decisiones de priorización o inversión?


## 🤔💬 Momento de reflexionar en lo aprendido


*Kahoot time!*

## 🚀 Para seguir aprendiendo :


- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta  nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [`Discord`](https://discord.com/channels/1081207584104656986/1420849538196836472).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal [`#project`](https://discord.com/channels/1081207584104656986/1420848813186351134) para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
    - En tus preguntas recuerda etiquetar a `@Dataconsulta` y ubica tu pregunta de acuerdo a `Sprint/Capitulo/Seccion`
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨